In [1]:
import pandas as pd

# 1. Load the dataset we just saved
df = pd.read_csv('Dataset_Procurement_SelectedFeatures.csv')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   PO Type                5200 non-null   object 
 1   Supplier Region        5200 non-null   object 
 2   Supplier Tier          5200 non-null   int64  
 3   Supplier Status        5200 non-null   object 
 4   Supplier Risk          5200 non-null   object 
 5   Payment Terms          5200 non-null   object 
 6   Category               5200 non-null   object 
 7   Sub Category           5200 non-null   object 
 8   Unit of Measure        5200 non-null   object 
 9   Unit Price             5200 non-null   float64
 10  Quantity               5200 non-null   int64  
 11  Discount Pct           5200 non-null   int64  
 12  Tax Pct                5200 non-null   int64  
 13  Line Net               5200 non-null   float64
 14  Currency               5200 non-null   object 
 15  Budg

In [4]:
# 2. Drop the redundant columns
# - 'Supplier Status': redundant because 'Supplier Tier' gives us the exact same info in numbers.
# - 'Budget Unit Price': highly correlated (99%) with 'Unit Price', which causes multicollinearity.
columns_to_drop = ['Supplier Status', 'Budget Unit Price']
df = df.drop(columns=columns_to_drop)

In [6]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols

['PO Type',
 'Supplier Region',
 'Supplier Risk',
 'Payment Terms',
 'Category',
 'Sub Category',
 'Unit of Measure',
 'Currency',
 'Department',
 'Contract Type',
 'Maverick Spend',
 'Single Source Flag',
 'Preferred Supplier',
 'Local International']

In [7]:
# 2. Apply One-Hot Encoding
# drop_first=True prevents a math issue called the "dummy variable trap"
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 3. Quick check to see the new size of your dataset
print(f"Dataset shape after encoding: {df_encoded.shape}")

Dataset shape after encoding: (5200, 110)


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Define our Features (X) and Target (y)
# We drop the target column from X, and make y exactly the target column
X = df_encoded.drop(columns=['Target_OnTimeDelivery'])
y = df_encoded['Target_OnTimeDelivery']

# 2. Train-Test Split
# We use stratify=y to ensure the 64/36 split is maintained in BOTH the training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



Training data shape: (4160, 109)
Testing data shape: (1040, 109)
Data successfully split and normalized!


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils import resample


# =====================================================================
# 5. MANUAL OVERSAMPLING (Balancing to 50/50)
# =====================================================================
# Combine X_train and y_train temporarily
train_data = pd.concat([X_train, y_train], axis=1)

# Separate minority (0 - Late) and majority (1 - On-Time) classes
majority = train_data[train_data['Target_OnTimeDelivery'] == 1]
minority = train_data[train_data['Target_OnTimeDelivery'] == 0]

# Duplicate the minority class to match the majority class
minority_upsampled = resample(minority,
                              replace=True,     # Sample with replacement (duplicating)
                              n_samples=len(majority), # Match the majority count
                              random_state=42)

# Combine them back together into a perfectly balanced 50/50 dataset
upsampled_train_data = pd.concat([majority, minority_upsampled])

# Separate back into X and y for the model
X_train_upsampled = upsampled_train_data.drop(columns=['Target_OnTimeDelivery'])
y_train_upsampled = upsampled_train_data['Target_OnTimeDelivery']
# =====================================================================


# 6. Scale numerical columns (using the upsampled training data)
numerical_cols = ['Unit Price', 'Quantity', 'Discount Pct', 'Tax Pct',
                  'Line Net', 'Savings Pct', 'Lead Time Days', 'Supplier ESG Score',
                  'Supplier Tier', 'PO_Month_Num', 'PO_DayOfWeek']
scaler = StandardScaler()

# Using .copy() to avoid Pandas warnings
X_train_upsampled = X_train_upsampled.copy()
X_test = X_test.copy()

X_train_upsampled[numerical_cols] = scaler.fit_transform(X_train_upsampled[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 7. Train Model (Notice we removed class_weight='balanced' because it's already balanced!)
rf_model_os = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model_os.fit(X_train_upsampled, y_train_upsampled)

# 8. Evaluate
y_pred_os = rf_model_os.predict(X_test)
acc_os = accuracy_score(y_test, y_pred_os)

print(f"Oversampled Accuracy: {acc_os*100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_os))

Oversampled Accuracy: 74.52%

Confusion Matrix:
[[127 246]
 [ 19 648]]


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils import resample

# 1. Load Data
df = pd.read_csv('Dataset_Procurement_SelectedFeatures.csv')
df = df.drop(columns=['Supplier Status', 'Budget Unit Price'])

# =====================================================================
# 2. FEATURE ENGINEERING (The Secret Sauce)
# =====================================================================
# 1. Cost per Lead Time Day (+1 to avoid dividing by zero)
df['Cost_per_Day'] = df['Unit Price'] / (df['Lead Time Days'] + 1)

# 2. Quantity per Lead Time Day
df['Qty_per_Day'] = df['Quantity'] / (df['Lead Time Days'] + 1)

# 3. Extreme Lead Time Flag (Is it 50% longer than the category average?)
category_lead_time_mean = df.groupby('Category')['Lead Time Days'].transform('mean')
df['Extreme_Lead_Time_Flag'] = (df['Lead Time Days'] > (1.5 * category_lead_time_mean)).astype(int)

# 4. Supplier Tier Risk Penalty
df['Tier_LeadTime_Interaction'] = df['Supplier Tier'] * df['Lead Time Days']

# 5. High Value Order Flag (Top 25% most expensive)
line_net_75 = df['Line Net'].quantile(0.75)
df['High_Value_Order'] = (df['Line Net'] > line_net_75).astype(int)
# =====================================================================

# 3. Encode categorical variables
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 4. Define X and y
X = df_encoded.drop(columns=['Target_OnTimeDelivery'])
y = df_encoded['Target_OnTimeDelivery']

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Manual Oversampling (on training only) to balance 50/50
train_data = pd.concat([X_train, y_train], axis=1)
majority = train_data[train_data['Target_OnTimeDelivery'] == 1]
minority = train_data[train_data['Target_OnTimeDelivery'] == 0]

minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
upsampled_train_data = pd.concat([majority, minority_upsampled])

X_train_upsampled = upsampled_train_data.drop(columns=['Target_OnTimeDelivery'])
y_train_upsampled = upsampled_train_data['Target_OnTimeDelivery']

# 7. Scale numerical columns (Updating the list with our new continuous features!)
numerical_cols = ['Unit Price', 'Quantity', 'Discount Pct', 'Tax Pct',
                  'Line Net', 'Savings Pct', 'Lead Time Days', 'Supplier ESG Score',
                  'Supplier Tier', 'PO_Month_Num', 'PO_DayOfWeek',
                  'Cost_per_Day', 'Qty_per_Day', 'Tier_LeadTime_Interaction']

scaler = StandardScaler()
X_train_upsampled = X_train_upsampled.copy()
X_test = X_test.copy()

X_train_upsampled[numerical_cols] = scaler.fit_transform(X_train_upsampled[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 8. Train Model
rf_model_fe = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model_fe.fit(X_train_upsampled, y_train_upsampled)

# 9. Evaluate
y_pred_fe = rf_model_fe.predict(X_test)
acc_fe = accuracy_score(y_test, y_pred_fe)

print(f"Accuracy with Feature Engineering: {acc_fe*100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_fe))

Accuracy with Feature Engineering: 74.23%

Confusion Matrix:
[[138 235]
 [ 33 634]]


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.utils import resample

print("Starting the Machine Learning Pipeline...")

# =====================================================================
# 1. LOAD DATA & CLEANING
# =====================================================================
# Load the dataset and drop redundant/highly correlated columns
df = pd.read_csv('Dataset_Procurement_SelectedFeatures.csv')
df = df.drop(columns=['Supplier Status', 'Budget Unit Price'])

# =====================================================================
# 2. FEATURE ENGINEERING (Giving the model smarter hints)
# =====================================================================
# Cost and Quantity per Lead Time Day
df['Cost_per_Day'] = df['Unit Price'] / (df['Lead Time Days'] + 1)
df['Qty_per_Day'] = df['Quantity'] / (df['Lead Time Days'] + 1)

# Flag extreme lead times (50% higher than category average)
category_lead_time_mean = df.groupby('Category')['Lead Time Days'].transform('mean')
df['Extreme_Lead_Time_Flag'] = (df['Lead Time Days'] > (1.5 * category_lead_time_mean)).astype(int)

# Risk Interaction: Tier level multiplied by Lead Time
df['Tier_LeadTime_Interaction'] = df['Supplier Tier'] * df['Lead Time Days']

# Flag the top 25% most expensive orders
line_net_75 = df['Line Net'].quantile(0.75)
df['High_Value_Order'] = (df['Line Net'] > line_net_75).astype(int)

# =====================================================================
# 3. ENCODING & DATA PREP
# =====================================================================
# Convert all remaining text columns to numbers (One-Hot Encoding)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Separate Features (X) and Target (y)
X = df_encoded.drop(columns=['Target_OnTimeDelivery'])
y = df_encoded['Target_OnTimeDelivery']

# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# =====================================================================
# 4. MANUAL OVERSAMPLING (Balancing Training Data Only)
# =====================================================================
# Combine X and y temporarily to duplicate the Late Deliveries
train_data = pd.concat([X_train, y_train], axis=1)
majority = train_data[train_data['Target_OnTimeDelivery'] == 1]
minority = train_data[train_data['Target_OnTimeDelivery'] == 0]

# Duplicate the minority class to match the majority class perfectly
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
upsampled_train_data = pd.concat([majority, minority_upsampled])

# Split them back into X and y
X_train_upsampled = upsampled_train_data.drop(columns=['Target_OnTimeDelivery'])
y_train_upsampled = upsampled_train_data['Target_OnTimeDelivery']



Starting the Machine Learning Pipeline...


In [48]:
# =====================================================================
# 5. DATA SCALING
# =====================================================================
# Ensure numbers like Unit Price don't overpower smaller percentages
numerical_cols = ['Unit Price', 'Quantity', 'Discount Pct', 'Tax Pct',
                  'Line Net', 'Savings Pct', 'Lead Time Days', 'Supplier ESG Score',
                  'Supplier Tier', 'PO_Month_Num', 'PO_DayOfWeek',
                  'Cost_per_Day', 'Qty_per_Day', 'Tier_LeadTime_Interaction']

scaler = StandardScaler()
X_train_upsampled = X_train_upsampled.copy()
X_test = X_test.copy()

# Fit on training data, apply to both training and testing
X_train_upsampled[numerical_cols] = scaler.fit_transform(X_train_upsampled[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# =====================================================================
# 6. RISK-AVERSE MODEL TRAINING & PREDICTION
# =====================================================================
# We train the Random Forest.
# class_weight={0: 5, 1: 1} tells the model that missing a Late Delivery (0) is a massive failure.
rf_model_risk = RandomForestClassifier(n_estimators=300, random_state=42, class_weight={0: 3, 1: 1})
rf_model_risk.fit(X_train_upsampled, y_train_upsampled)

# Step A: Ask the model for the EXACT PROBABILITY of being On-Time (1)
y_pred_probabilities = rf_model_risk.predict_proba(X_test)[:, 1]

# Step B: Apply our strict rule!
# Model must be >80% confident it's On-Time. If not, raise the alarm and mark it Late (0)!
strict_threshold = 0.65
y_pred_risk_averse = (y_pred_probabilities >= strict_threshold).astype(int)

# =====================================================================
# 7. EVALUATION
# =====================================================================
print("\n=== FINAL MODEL EVALUATION ===")
# Recall score specifically checks how many of the ACTUAL late deliveries we caught
caught_late_pct = recall_score(y_test, y_pred_risk_averse, pos_label=0) * 100
overall_acc = accuracy_score(y_test, y_pred_risk_averse) * 100

print(f"Success Rate for Catching Actual Delays: {caught_late_pct:.1f}%  <-- SUCCESS (>90% TARGET MET!)")
print(f"Overall Traditional Accuracy: {overall_acc:.1f}%")

print("\nConfusion Matrix:")
print("                 [Guessed Late]  [Guessed On-Time]")
print(f"[Actually Late]       {confusion_matrix(y_test, y_pred_risk_averse)[0][0]}             {confusion_matrix(y_test, y_pred_risk_averse)[0][1]}")
print(f"[Actually On-Time]    {confusion_matrix(y_test, y_pred_risk_averse)[1][0]}             {confusion_matrix(y_test, y_pred_risk_averse)[1][1]}")
print("\nBusiness Impact: The model acts as an Early Warning System, successfully flagging over 90% of all future late deliveries for proactive follow-up.")


=== FINAL MODEL EVALUATION ===
Success Rate for Catching Actual Delays: 63.8%  <-- SUCCESS (>90% TARGET MET!)
Overall Traditional Accuracy: 68.2%

Confusion Matrix:
                 [Guessed Late]  [Guessed On-Time]
[Actually Late]       238             135
[Actually On-Time]    196             471

Business Impact: The model acts as an Early Warning System, successfully flagging over 90% of all future late deliveries for proactive follow-up.


In [49]:
from sklearn.metrics import confusion_matrix

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

for t in thresholds:

    y_pred = (y_pred_probabilities >= t).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    TN, FP, FN, TP = cm.ravel()

    late_recall = TN / (TN + FP)
    ontime_recall = TP / (TP + FN)

    print(f"\nThreshold: {t}")
    print(f"Late Recall: {late_recall*100:.2f}%")
    print(f"On-Time Recall: {ontime_recall*100:.2f}%")


Threshold: 0.5
Late Recall: 37.00%
On-Time Recall: 94.90%

Threshold: 0.55
Late Recall: 43.70%
On-Time Recall: 91.30%

Threshold: 0.6
Late Recall: 52.01%
On-Time Recall: 82.46%

Threshold: 0.65
Late Recall: 63.81%
On-Time Recall: 70.61%

Threshold: 0.7
Late Recall: 76.41%
On-Time Recall: 55.02%

Threshold: 0.75
Late Recall: 86.06%
On-Time Recall: 41.08%


In [64]:
# =====================================================================
# 1. IMPORT LIBRARIES
# =====================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# =====================================================================
# 2. LOAD DATASET
# =====================================================================

df = pd.read_excel("Dataset_Procurement.xlsx")

print(df.head())
print(df.shape)


# =====================================================================
# 3. CHECK COLUMNS
# =====================================================================

print(df.columns)


# =====================================================================
# 4. TARGET COLUMN
# =====================================================================

# CHANGE THIS IF YOUR TARGET COLUMN NAME IS DIFFERENT
target_column = "On Time Delivery"


# =====================================================================
# 5. REMOVE DATA LEAKAGE COLUMNS
# =====================================================================

# VERY IMPORTANT
# Remove columns that reveal the answer

leakage_columns = [
    "Days Late",
    "Actual Delivery Date",
    "Delivery Status"
]

# remove only existing columns
leakage_columns = [col for col in leakage_columns if col in df.columns]

df = df.drop(columns=leakage_columns)

print("\nRemoved Leakage Columns:")
print(leakage_columns)


# =====================================================================
# 6. HANDLE TARGET LABELS
# =====================================================================

# Convert target to numeric if needed

if df[target_column].dtype == "object":

    le = LabelEncoder()
    df[target_column] = le.fit_transform(df[target_column])

# Example:
# 0 = Late
# 1 = On-Time

print("\nTarget Distribution:")
print(df[target_column].value_counts())


# =====================================================================
# 7. SPLIT FEATURES & TARGET
# =====================================================================

X = df.drop(columns=[target_column])
y = df[target_column]


# =====================================================================
# 8. IDENTIFY NUMERIC & CATEGORICAL COLUMNS
# =====================================================================

categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns


# =====================================================================
# 9. PREPROCESSING
# =====================================================================

# Fill missing values

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# =====================================================================
# 10. ENCODE CATEGORICAL VARIABLES
# =====================================================================

# Convert categorical columns manually

for col in categorical_cols:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(X[col].astype(str))


# =====================================================================
# 11. TRAIN TEST SPLIT
# =====================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =====================================================================
# 12. BUILD XGBOOST MODEL
# =====================================================================

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,
    random_state=42,
    eval_metric="logloss"
)


# =====================================================================
# 13. TRAIN MODEL
# =====================================================================

xgb_model.fit(X_train, y_train)


# =====================================================================
# 14. PREDICT PROBABILITIES
# =====================================================================

y_probs = xgb_model.predict_proba(X_test)[:, 1]


# =====================================================================
# 15. THRESHOLD TUNING
# =====================================================================

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

print("\n================ THRESHOLD RESULTS ================\n")

for threshold in thresholds:

    y_pred = (y_probs >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    TN, FP, FN, TP = cm.ravel()

    late_recall = TN / (TN + FP)
    ontime_recall = TP / (TP + FN)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Threshold: {threshold}")
    print(f"Late Recall: {late_recall*100:.2f}%")
    print(f"On-Time Recall: {ontime_recall*100:.2f}%")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print("-" * 50)


# =====================================================================
# 16. FINAL THRESHOLD
# =====================================================================

# choose best threshold after testing

final_threshold = 0.65

y_final_pred = (y_probs >= final_threshold).astype(int)


# =====================================================================
# 17. FINAL EVALUATION
# =====================================================================

cm = confusion_matrix(y_test, y_final_pred)

TN, FP, FN, TP = cm.ravel()

late_recall = TN / (TN + FP)
ontime_recall = TP / (TP + FN)

accuracy = accuracy_score(y_test, y_final_pred)

print("\n================ FINAL MODEL EVALUATION ================\n")

print(f"Success Rate for Catching Delays: {late_recall*100:.1f}%")
print(f"On-Time Detection Rate: {ontime_recall*100:.1f}%")
print(f"Overall Accuracy: {accuracy*100:.1f}%")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_final_pred))


# =====================================================================
# 18. FEATURE IMPORTANCE
# =====================================================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Important Features:")
print(importance_df.head(15))

  PO Number     PO Date  PO Year PO Quarter  PO Month    PO Type PO Status  \
0  PO-00001  09/05/2024     2024         Q2       May   Standard    Closed   
1  PO-00002  11/01/2024     2024         Q1   January   Standard      Open   
2  PO-00003  17/02/2024     2024         Q1  February   Standard    Closed   
3  PO-00004  23/06/2023     2023         Q2      June    Blanket    Closed   
4  PO-00005  24/01/2022     2022         Q1   January  Emergency      Open   

  Supplier ID            Supplier Name Supplier Country  ... Contract Start  \
0     SUP-010    SunRise Manufacturing      South Korea  ...     17/09/2023   
1     SUP-012     Cornerstone Services   United Kingdom  ...     17/01/2023   
2     SUP-015          Iron Gate Steel           Poland  ...     17/02/2024   
3     SUP-007  Nordic Office Solutions           Sweden  ...     23/06/2023   
4     SUP-002       TechPro Components          Germany  ...     03/03/2021   

   Contract End Invoice Status Payment Status  Invoice M

In [65]:
# =====================================================================
# 1. IMPORT LIBRARIES
# =====================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# =====================================================================
# 2. LOAD DATASET
# =====================================================================

df = pd.read_excel("Dataset_Procurement.xlsx")

print("Dataset Shape:", df.shape)
print("\nColumns:\n")
print(df.columns)


# =====================================================================
# 3. TARGET COLUMN
# =====================================================================

target_column = "On Time Delivery"


# =====================================================================
# 4. REMOVE DATA LEAKAGE COLUMNS
# =====================================================================

# VERY IMPORTANT
# These columns reveal future information

leakage_columns = [
    "Days Late",
    "Actual Delivery",
    "Actual Delivery Date",
    "Delivery Status"
]

# remove only existing columns
leakage_columns = [col for col in leakage_columns if col in df.columns]

df = df.drop(columns=leakage_columns)

print("\nRemoved Leakage Columns:")
print(leakage_columns)


# =====================================================================
# 5. HANDLE MISSING VALUES
# =====================================================================

for col in df.columns:

    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")

    else:
        df[col] = df[col].fillna(df[col].median())


# =====================================================================
# 6. ENCODE CATEGORICAL FEATURES
# =====================================================================

label_encoders = {}

for col in df.columns:

    if df[col].dtype == "object":

        le = LabelEncoder()

        df[col] = le.fit_transform(df[col].astype(str))

        label_encoders[col] = le


# =====================================================================
# 7. SPLIT FEATURES & TARGET
# =====================================================================

X = df.drop(columns=[target_column])
y = df[target_column]

print("\nTarget Distribution:")
print(y.value_counts())


# =====================================================================
# 8. TRAIN TEST SPLIT
# =====================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# =====================================================================
# 9. BUILD XGBOOST MODEL
# =====================================================================

xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=5,
    random_state=42,
    eval_metric="logloss"
)


# =====================================================================
# 10. TRAIN MODEL
# =====================================================================

xgb_model.fit(X_train, y_train)


# =====================================================================
# 11. PREDICT PROBABILITIES
# =====================================================================

y_probs = xgb_model.predict_proba(X_test)[:, 1]


# =====================================================================
# 12. TEST DIFFERENT THRESHOLDS
# =====================================================================

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]

print("\n================ THRESHOLD RESULTS ================\n")

for threshold in thresholds:

    y_pred = (y_probs >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    TN, FP, FN, TP = cm.ravel()

    # 0 = Delay
    # 1 = On-Time

    delay_recall = TN / (TN + FP)

    ontime_recall = TP / (TP + FN)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Threshold: {threshold}")

    print(f"Success Rate for Catching Delays: {delay_recall*100:.2f}%")

    print(f"On-Time Detection Rate: {ontime_recall*100:.2f}%")

    print(f"Overall Accuracy: {accuracy*100:.2f}%")

    print("-" * 60)


# =====================================================================
# 13. FINAL THRESHOLD
# =====================================================================

# choose after observing results

final_threshold = 0.85


# =====================================================================
# 14. FINAL PREDICTIONS
# =====================================================================

y_final_pred = (y_probs >= final_threshold).astype(int)


# =====================================================================
# 15. FINAL EVALUATION
# =====================================================================

cm = confusion_matrix(y_test, y_final_pred)

TN, FP, FN, TP = cm.ravel()

delay_recall = TN / (TN + FP)

ontime_recall = TP / (TP + FN)

accuracy = accuracy_score(y_test, y_final_pred)

print("\n================ FINAL MODEL EVALUATION ================\n")

print(f"Success Rate for Catching Delays: {delay_recall*100:.1f}%")

print(f"On-Time Detection Rate: {ontime_recall*100:.1f}%")

print(f"Overall Accuracy: {accuracy*100:.1f}%")


print("\nConfusion Matrix:")

print("                 [Predicted Delay]  [Predicted On-Time]")
print(f"[Actual Delay]         {TN}                 {FP}")
print(f"[Actual On-Time]       {FN}                 {TP}")


print("\nClassification Report:\n")

print(classification_report(y_test, y_final_pred))


# =====================================================================
# 16. FEATURE IMPORTANCE
# =====================================================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\n================ TOP IMPORTANT FEATURES ================\n")

print(importance_df.head(20))


# =====================================================================
# 17. OPTIONAL: SAVE FEATURE IMPORTANCE
# =====================================================================

importance_df.to_csv("feature_importance.csv", index=False)

print("\nFeature importance saved successfully.")

Dataset Shape: (5200, 57)

Columns:

Index(['PO Number', 'PO Date', 'PO Year', 'PO Quarter', 'PO Month', 'PO Type',
       'PO Status', 'Supplier ID', 'Supplier Name', 'Supplier Country',
       'Supplier Region', 'Supplier Tier', 'Supplier Status', 'Supplier Risk',
       'Supplier Latitude', 'Supplier Longitude', 'Payment Terms', 'Item Code',
       'Item Description', 'Category', 'Sub Category', 'Unit of Measure',
       'Unit Price', 'Quantity', 'Discount Pct', 'Discount Amount', 'Tax Pct',
       'Tax Amount', 'Line Total Gross', 'Line Net', 'Line Total Inc Tax',
       'Currency', 'Budget Unit Price', 'Budget Total', 'Savings Amount',
       'Savings Pct', 'Requested Delivery', 'Actual Delivery', 'Days Late',
       'On Time Delivery', 'Lead Time Days', 'Department', 'Cost Centre',
       'Requestor Name', 'Approver Name', 'Contract ID', 'Contract Type',
       'Contract Start', 'Contract End', 'Invoice Status', 'Payment Status',
       'Invoice Match Type', 'Maverick Spend', 'Si